In [ ]:
# Cargo Paquetes
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import statsmodels.api as sm

In [ ]:
# Cargo Base de Datos
### Importar información
from google.colab import drive
drive.mount('/content/drive')

## Carga de Informacion
ruta = "/content/drive/MyDrive/Metodos Cuantitativos y Analisis de Datos/Datos/EJ.5.6_1.csv"
datos = pd.read_csv(ruta, sep = ",")
print(datos.head())

# Estadisticas Descriptivas
print("\nMedia de ct:", datos["ct"].mean())
print("Media de q :", datos["q"].mean())
print("Cor(ct, q):", datos["ct"].corr(datos["q"]))

# Gráfico de Relación Lineal
plt.figure(figsize=(6,5))
sns.scatterplot(data=datos, x="q", y="ct")
sns.regplot   (data=datos, x="q", y="ct", ci=False, scatter=False, color="red")
plt.title("Precio vs Tamaño (m²)")
plt.show()

In [ ]:
### Modelo de Regresión Lineal
mod = smf.ols("ct ~ q", data=datos).fit()
print(mod.summary())

## Gráfico de ajuste
datos["y_hat"] = mod.fittedvalues
datos["resid"] = mod.resid

plt.figure(figsize=(6,4))
plt.scatter(datos["q"], datos["ct"], label="Datos", alpha=0.7)
plt.plot(datos["q"], datos["y_hat"], color="navy", label="Recta OLS")
for _, row in datos.iterrows():
    plt.plot([row["q"], row["q"]], [row["ct"], row["y_hat"]], color="gray", lw=0.6)
plt.xlabel("q")
plt.ylabel("ct")
plt.title("Regresión Lineal Simple")
plt.legend()
plt.tight_layout()
plt.show()

### Evaluación del Modelo
## Genero Tabla para Evaluación
#Prediccion y Métricas de Influencia
pred_df = mod.get_prediction(datos).summary_frame()
infl_df = mod.get_influence().summary_frame()

# Uno
aug = pd.concat([
    datos.reset_index(drop=True),
    pred_df[["mean"]].rename(columns={"mean":   "fitted"}),
    infl_df[["standard_resid", "student_resid", "hat_diag", "cooks_d"]]],
    axis=1)
print(aug)

# Genero Variable de Error de Predicción y Orden de Observaciones
aug["resid"] = aug["ct"] - aug["fitted"]
aug["id"]    = np.arange(1, len(aug)+1)

## Diagnóstico del Modelo
# Residuos vs Predichos (especificación y homoscedasticidad)
plt.figure(figsize=(6,4))
sns.scatterplot(x=aug["fitted"], y=aug["resid"])
plt.axhline(0, ls="--", color="gray")
plt.xlabel("Valores ajustados")
plt.ylabel("Residuos")
plt.title("Residuos vs Ajustados")
plt.show()

# Q-Q Plot (Normalidad)
sm.qqplot(aug["standard_resid"], line="45")
plt.title("QQ-plot residuos estandarizados")
plt.show()

# Tratamiento de Observaciones
# Leverage (palanca)
print(aug.loc[aug["hat_diag"] > 6/len(aug), ["id", "hat_diag"]].sort_values("hat_diag", ascending=False))
# Influyentes
print(aug.loc[aug["cooks_d"] >= 1, ["id", "cooks_d"]].sort_values("cooks_d", ascending=False))
# Atípicos
print(aug.loc[aug["standard_resid"].abs() > 3, ["id", "standard_resid"]].sort_values("standard_resid", ascending=False))

### Estrategia (dado que existe una observación influyente, esta variable la quitamos de la base de datos)
# Efecto de Quitar la Observación 21
fig, ax = plt.subplots(figsize=(6,5))
sns.scatterplot(x="q", y="ct", data=datos,        label="Datos",    ax=ax)
sns.regplot( x="q", y="ct", data=datos,        ci=False, color="black", label="Completo", ax=ax)
sns.regplot(x="q", y="ct", data=datos.drop(20), ci=False, color="red",   label="Sin obs21", ax=ax)
ax.legend(); ax.set_title("Impacto de la obs 21"); plt.show()



In [ ]:
### Modelo sin Observación 21
# Estimación
datos_no21 = datos.drop(index=datos.index[20])
mod2 = smf.ols("ct ~ q", data=datos_no21).fit()
print(mod2.summary())

# Comparación
tidy1 = mod.summary2().tables[1].reset_index().rename(columns={"index":"term"})
tidy2 = mod2.summary2().tables[1].reset_index().rename(columns={"index":"term"})
comp = pd.concat([
    tidy1.assign(model="con_obs21"),
    tidy2.assign(model="sin_obs21")
], ignore_index=True)
print("\nCoeficientes comparados:")
print(comp[["model","term","Coef.","Std.Err.","P>|t|"]])

### Evaluación del Modelo
## Genero Tabla para Evaluación

#Prediccion y Métricas de Influencia
pred2 = mod2.get_prediction(datos_no21).summary_frame()
infl2 = mod2.get_influence().summary_frame()
aug2 = pd.concat([
    datos_no21.reset_index(drop=True),
    pred2[["mean"]].rename(columns={"mean":"fitted"}),
    infl2[["standard_resid","hat_diag","cooks_d"]]
], axis=1)
aug2["resid"] = aug2["ct"] - aug2["fitted"]
aug2["id"]    = np.arange(1, len(aug2)+1)

## Diagnóstico del Modelo
# Residuos vs Predichos (especificación y homoscedasticidad)
plt.figure(figsize=(6,4))
sns.scatterplot(x=aug2["fitted"], y=aug2["resid"])
plt.axhline(0, ls="--", color="gray")
plt.title("Residuos vs Ajustados (sin obs 21)")
plt.show()

# Q-Q Plot (Normalidad)
sm.qqplot(aug2["standard_resid"], line="45")
plt.title("QQ-plot residuos estandarizados (sin obs 21)")
plt.show()

# Tratamiento de Observaciones
# Leverage (palanca)
print(aug2.loc[aug2["hat_diag"] > 6/len(aug2), ["id", "hat_diag"]].sort_values("hat_diag", ascending=False))
# Influyentes
print(aug2.loc[aug2["cooks_d"] >= 1, ["id", "cooks_d"]].sort_values("cooks_d", ascending=False))
# Atípicos
print(aug2.loc[aug2["standard_resid"].abs() > 3, ["id", "standard_resid"]].sort_values("standard_resid", ascending=False))